# Acquiring And Storing  Data In Python

This lab is targeted at students who are familiar with Python who are looking to query an API and store data into a database.

## Querying APIs Using Python

Python is a very powerful language that can quickly be used to ingest data from an Application Program Interface (API) and make it available for consumption later by storing it in a database. APIs specify how software components interact. 

There are different methods that are used to interact with HTTP APIs.

| Method | Type           | Description                                                       |
|--------|----------------|-------------------------------------------------------------------|
| POST   | Create         | Used to create data                                               |
| GET    | Read           | Used to retrieve data                                             |
| PUT    | Update/Modify  | Updates the entity if it exists / creates a new one if it doesn't | 
| PATCH  | Update/Replace | Applies a partial update to data                                  |
| DELETE | Remove         | Deletes data                                                      |

## **Lab:** Querying an API Using Python

In this lab we will query an API in Python, and extract specific fields to print.  Data can be exchanged in any format but we will receive data in [Java Script Object Notation (JSON)](https://www.json.org/json-en.html) which allows for ease of understanding.

Let's take a look at a quick example of how to query an API using python. We'll be querying a dad joke API, which will return a random dad joke (along with other some other data). Take a look at the example below and try to understand how program works. We'll go over the inner workings of it afterwards.

**NOTE**: Because we're querying a random fact, the results commented in the code will be different when actually ran.

In [90]:
import requests #1

headers = {'Accept': 'application/json'} #2 
response = requests.get('https://icanhazdadjoke.com/', headers=headers) #3

print(response) #4
# <Response [200]>

data = response.json() #5

print(data) #6
# {"id": "FdNZTnWvHBd", "joke": "I’ve just been reading a book about anti-gravity, it’s impossible to put down!", "status": 200}

print(data['joke']) #7
# I’ve just been reading a book about anti-gravity, it’s impossible to put down!

<Response [200]>
{'id': 'VDdxcp4ob', 'joke': 'How do you make Lady Gaga cry? Poker face. ', 'status': 200}
How do you make Lady Gaga cry? Poker face. 


#### Diving into our code, let's go over how this all works.
- Notice that line `#1` imports the `requests` library. This is a very common python library used HTTP requests, and is the backbone of our API consumer.

- The request headers are set on line `#2`. This is a dictionary that contains the headers that will be sent to the API. In this case, we're specifying that we want to receive JSON data back from the API.

- At line `#3` is where the actual querying of the API occurs. The `requests` package will connect to the URL and include the prepared headers, retrieve the data and store it in an easy-to-use python object. This data is then store in a variable called `response`

- Line `#4` prints the result of our `request` call. The variable `response` if of type `Response` and shows us one value, 200. This is the status code from the HTTP server. A status of 200 means that everything went smoothly.

- Now we need to actually work with the data that we received from the dad joke web API. This is done on line `#5`. The `requests` package makes working with JSON data extremely easy, by simply invoking the `.json()` method. This will in return, provide a JSON object of the result (if the result was written in JSON format, otherwise raise an error). Since we know JSON objects are key/value based, this becomes a dictionary type in python, which makes retrieving information a piece of cake.

- The raw JSON (dictionary) data is printed on line `#6`. This shows all the data we have to work with.

- From the API, the `joke` field contains the joke. The other fields, such as `id`, provide extra information about the joke. Line `#7` prints out the joke to the console.

This can be simplified into a Bash one-line that you can run:
```bash
python -c "import requests; print(requests.get('https://icanhazdadjoke.com/', headers={'Accept': 'application/json'}).json()['joke'])"
```

In [66]:
!python -c "import requests; print(requests.get('https://icanhazdadjoke.com/', headers={'Accept': 'application/json'}).json()['joke'])"

What is a witch's favorite subject in school? Spelling!


---
## Storing Data from an API

Now that we have the basic understanding of how to query an API, let's expand our knowledge to query an API to build a database.
We're going to build a rudimentary book catalog.
To build the database, we will query the [Open Books Library API](https://openlibrary.org/dev/docs/api/books), which contains a lot of information about a book based on the books ISBN number. 

### The Database Model

The database will have a very simple design, with the main entities being _Books_, _Authors_, and _Themes_.
A _book_ will consist of an ISBN, title, subtitle, and number of pages.
Remember it is proper to have a primary key that is unique to identify the row.
Fortunately, an ISBN is a unique identifier for a book, so we'll use that as our primary key.
If you're thinking, _"But isn't there an ISBN 10 and an ISBN 13 that can reference the same book?"_, you're not wrong
However, for the sake of brevity, only the ISBN queried will be used.
An _author_ will consist of the author's name and an `id` that will increment every time a new author is inserted into the table.
Finally, a _theme_ has a similar schema as an author, it has the name of the theme, and an automatically incrementing `id`.

The last part is tying our data together.
A book may have one to many authors, and as well, an author may have one to many books.
This type of relationship is called a [many-to-many relationship](https://en.wikipedia.org/wiki/Many-to-many_(data_model)).
To map the _book_ and _author_ entities together, we use a [link table](https://en.wikipedia.org/wiki/Associative_entity).
Using a link table helps reduce duplicate data within a table.
For example, let's look at the book: [Design Patterns: Elements of Reusable Object Oriented Software](https://en.wikipedia.org/wiki/Design_Patterns).
This book has four authors associated with it.
If we had included an `author` column in the _book_ table, there would be multiple rows with the same title, isbn, subtitle, etc.
The only difference would be the author's name was changed.
Ignoring the fact that the ISBN in our table must be unique, you can see how adding in many books can result in mounds of duplicate data, and this is not desirable.
The same principle is applied to a _book_ and _theme_ relationship.

For a graphical representation, refer to the entity-relationship diagram below:

![image of our data models](./erd-book_catalog.png)

---
## **Lab:** Putting it together by reading data from an API and Storing it into a database.

**NOTE**: The tables should ideally be created in SQL Management Studio. The code below is just FYI

### Create the database

Create the database in SQL Server Management Studio. The code is reproduced below:

```sql
USE master
GO

if exists (select * from sysdatabases where name='book_catalog')
	ALTER DATABASE book_catalog SET SINGLE_USER WITH  ROLLBACK IMMEDIATE
    drop database book_catalog

GO
CREATE DATABASE book_catalog

GO

```




### Create the tables

Below is the code for creating the tables in SQL Management Studio.
If you're not sure what everything means, don't fret.
Just make sure you can see how the fields in the diagram above match the fields within the `CREATE TABLE` statements.

```sql
USE book_catalog
GO

-- CREATE TABLE books
CREATE TABLE books (
    isbn VARCHAR(13) NOT NULL,
    title NVARCHAR(100) NOT NULL,
    subtitle NVARCHAR(100) DEFAULT NULL,
    no_pages INT DEFAULT 0,
    PRIMARY KEY (isbn)
)
GO

-- CREATE TABLE authors
CREATE TABLE authors (
    id INT IDENTITY (1, 1) NOT NULL,
    name NVARCHAR(30) NOT NULL,
    PRIMARY KEY (id)
)
GO

-- CREATE TABLE themes
CREATE TABLE themes(
    id INT IDENTITY (1, 1) NOT NULL,
    name NVARCHAR(100) NOT NULL,
    PRIMARY KEY (id)
)
GO

-- CREATE TABLE authors_books
CREATE TABLE authors_books (
    isbn VARCHAR(13),
    author_id INT,
    PRIMARY KEY (isbn, author_id),
    FOREIGN KEY (isbn) REFERENCES books(isbn),
    FOREIGN KEY (author_id) REFERENCES authors(id)
)
GO

-- CREATE TABLE books_themes
CREATE TABLE books_themes (
    isbn VARCHAR(13),
    theme_id INT,
    PRIMARY KEY (isbn, theme_id),
    FOREIGN KEY (isbn) REFERENCES books(isbn),
    FOREIGN KEY (theme_id) REFERENCES themes(id)
)
GO
```


## Now to build a new Python notebook that will automate the process.

The script will execute the following:
- Connect to the database
- For each ISBN in a list of ISBNs:
  - Query the OpenLibrary API to retrieve the book data
  - Manipulate the returned JSON data into python variables
  - Insert the data into the database

### Connect to the database

Create a Python notebook by clicking File > New notebook

Name the notebook `book-catalog`. Next, fill in the empty code blocks by adding in the required libraries, and the connection to the database.

**NOTE**: Change the `server` variable below to match your SQL Server Name

In [123]:
# server name & database
SERVER = 'DESKTOP-DUVEH8J'  # Your server name
DATABASE = 'book_catalog'   # the database you're connecting to

# Create the connection string using Windows Authentication
connection_string = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SERVER};DATABASE={DATABASE};Trusted_Connection=yes;'

#### Test the connection:

In [36]:
import pyodbc

# Try to establish a connection
try:
    db = pyodbc.connect(connection_string)
    cursor = db.cursor()
    print("Connection successful!")

    # Close the connection
    db.close()

except pyodbc.Error as e:
    print("Error while connecting to SQL Server:", e)

Connection successful!


### Set up the Data for the API

Next, create a list of ISBNs to be queried against the API. The `API_URL` variable is the base URL for the OpenLibrary API, and afterwards, the parameters can be added in.

```python
# the base URL to query the books API
API_URL = 'http://openlibrary.org/api/books'

# a list of ISBNs to be added to the catalog
isbn_list = [
    '978-0201853926',
    '0201558025',
    '978-1-93435-645-6',
    '9780199218462',
    '978-4915512377',
    '1593278551',
    '0811862151',
    '1844834115',
    '9780545790352',
    '978-0201633610',
]
```

### Query the OpenLibrary API

For each ISBN in the list, we build an `isbn_payload`.
This is the format in which the OpenLibrary API requires the `bibkeys` parameter to be formatted.
For example, using the book from before, the `isbn_payload` would be set to `ISBN:0201633612`.
Next, set up the parameters required for the API to retrieve the book's data.
We set the `format` to return JSON, and the `jscmd` to _data_.
With all the parameters set, use the `requests` package to make the call to the API.

```python
# cycle through the list and query the API
for isbn in isbn_list:
    # grab a fresh cursor
    cursor = db.cursor()

    # set up proper search parameters for the API
    isbn_payload = 'ISBN:%s' % isbn

    # set all parameters needed to make the request
    params = {
        'bibkeys':isbn_payload,
        'format':'json',
        'jscmd':'data'
    }

    # send the request
    r = requests.get(API_URL, params)
```

Go to: https://openlibrary.org/api/books?bibkeys=ISBN:978-0201633610&format=json&jscmd=data to view the raw JSON output from the API.

#### Use Python to see if the API is running

In [80]:
# the base URL to query the books API
API_URL = 'https://openlibrary.org/api/books'

# set all parameters needed to make the request
isbn_payload = 'ISBN:978-0201633610' 
params = {
    'bibkeys':isbn_payload,
    'format':'json',
    'jscmd':'data'
}
print(f'{API_URL}?bibkeys={params['bibkeys']}&format={params['format']}&jscmd={params['jscmd']}')
   
# send the request
try:
    r = requests.get(API_URL, params)
    print(r.json()[isbn_payload])
except requests.exceptions.HTTPError as e:
    print(f'An Http Error occurred:\n{e}')
except requests.exceptions.ConnectionError as e:
    print(f'An Error Connecting to the API occurred:\n{e}')
except requests.exceptions.Timeout as e:
    print(f'A Timeout Error occurred:\n{e}')
except requests.exceptions.RequestException as e:
    print(f'An Unknown Error occurred:\n{e}')

https://openlibraryXX.org/api/books?bibkeys=ISBN:978-0201633610&format=json&jscmd=data
An Error Connecting to the API occurred:
HTTPSConnectionPool(host='openlibraryxx.org', port=443): Max retries exceeded with url: /api/books?bibkeys=ISBN%3A978-0201633610&format=json&jscmd=data (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002185CD463F0>: Failed to resolve 'openlibraryxx.org' ([Errno 11001] getaddrinfo failed)"))


### Retreiving the Book and Manipulating the Data

To retrieve the book data, we convert the response from the API into JSON, and access the object data for the key `isbn_payload`.
Which if you recall is ISBN:0201633612 for our example book.
From there we can obtain all the information we need from the JSON data.
For each piece of information we need, check that it is in the response and then assign it to a variable.
For the `authors` and `themes`, the python code creates a list of strings containing only the text data from the JSON.
Finally, the `isbn_clean` variable removes any extra characters and leaves only digits to meet the ISBN VARCHAR(13) requirement.

```python
    # retreive the book from the JSON data
    book = r.json()[isbn_payload]

    # obtain all information we require to put into our database
    title = book['title'][:100]
    subtitle = book['subtitle'][:100] if 'subtitle' in book else ''
    no_pages = book['number_of_pages'] if 'number_of_pages' in book else 0
    authors = [auth['name'] for auth in book['authors']]
    themes = [subj['name'] for subj in book['subjects']] if 'subjects' in book else []

    # clean up ISBN from input (numbers only to fit the VARCHAR(13) limit)
    isbn_clean = re.sub('[^0-9]', '', isbn)
```

**Note**: We limit the size of the data to 100 Because the column length in the databse tables is set to 100

### Insert Data into the Database

#### Book data
Inserting the book data is very straight forward.
Take the data from earlier and execute an `INSERT` statement.

```python
    # insert data into database

    # insert book
    insert_book_stmt = 'INSERT INTO books(isbn, title, subtitle, no_pages) VALUES( %s, %s, %s, %s )'
    cursor.execute(insert_book_stmt, (isbn_clean, title, subtitle, no_pages))
```


#### Authors
To insert the authors into the database is a bit more complex.
Each author of the book needs to be inserted in the database, and then it needs to be added into the `authors_books` table as well.

However, we can't blindly insert each author into the database, otherwise we would have duplicate author names with different ids, rendering our link table useless.
To solve this issue, query the `authors` table first and check if the author already exists.
If it doesn't, insert the new author and use `cursor.lastrowid` to get the newly inserted author's id.

However, if the author does exist, retrieve the `id` from the SELECT statement.
Now that the hard part is out of the way, the link table data can be inserted as simply as the book.

```python
    # insert authors
    # check if author exists in DB otherwise create and retreive the ID
    insert_author_stmt = 'INSERT INTO authors(author_name) VALUES(?)'
    insert_authors_books_stmt = 'INSERT INTO authors_books VALUES (?, ?)'
    for author in authors:
        print(f' - Author: {author}')
        select_author_id_stmt = 'SELECT id FROM authors WHERE author_name = ?;'
        cursor.execute(select_author_id_stmt, (author))
        result = cursor.fetchone()

        if result is None:
            cursor.execute(insert_author_stmt, (author))
            # Get the ID of the last inserted row using SCOPE_IDENTITY()
            cursor.execute('SELECT SCOPE_IDENTITY();')
            author_id = cursor.fetchone()[0]
        else:
            author_id = result[0]

        # insert author_id into authors_books
        try:
            cursor.execute(insert_authors_books_stmt, (isbn_clean, author_id))
        except pyodbc.IntegrityError as e:
            pass
```

**NOTE**: If you are using the `pyodbc` library for SQL Server, the parameterized query approach with `?` is safe against SQL injection as the user input is treated as data, not executable SQL.


#### Themes
To add themes into the database, follow the same procedure as the authors.

```python
    # insert themes
    # check if theme exists in DB otherwise create and retreive the ID
    insert_theme_stmt = 'INSERT INTO themes(theme_name) VALUES(?)'
    insert_books_themes_stmt = 'INSERT INTO books_themes VALUES (?, ?)'
    for theme in themes:
        print(f' - Theme: {theme}')
        select_theme_id_stmt = 'SELECT id FROM themes WHERE theme_name = ?;'
        cursor.execute(select_theme_id_stmt, (theme))
        result = cursor.fetchone()

        if result is None:
            cursor.execute(insert_theme_stmt, theme)
            # Get the ID of the last inserted row using SCOPE_IDENTITY()
            cursor.execute('SELECT SCOPE_IDENTITY();')
            theme_id = cursor.fetchone()[0]
        else:
            theme_id = result[0]

        # insert record into books_themes
        try:
            cursor.execute(insert_books_themes_stmt, (isbn_clean, theme_id))
        except pyodbc.IntegrityError as e:
            pass
```


#### commit and close the cursor
Finally, before moving on to the next ISBN in the list, commit the changes to the database and close the cursor.

```python
    db.commit()
    cursor.close()
```

---
## Delete existing data (optional)

The Python code is written is such a way that it doesn't matter if there is existing data in the tables.

But if you want to you can run the code below to delete all existing data from the database:

In [145]:
db = pyodbc.connect(connection_string)
cursor = db.cursor()

# Use the book_catalog database
cursor.execute('USE book_catalog')

# Delete data
cursor.execute('DELETE FROM authors_books')
cursor.execute('DELETE FROM authors_books')
cursor.execute('DELETE FROM books_themes')
cursor.execute('DELETE FROM themes')
cursor.execute('DELETE FROM books')
cursor.execute('DELETE FROM authors')

# Commit and close the cursor
db.commit()
cursor.close()

---
## **Solution**: Python Code in its Entirety

In [68]:
import re
import pyodbc
import requests

# server name & database
SERVER = 'DESKTOP-DUVEH8J' # This must be YOUR server name
DATABASE = 'book_catalog'  # the database you're connecting to

# Create the connection string using Windows Authentication
connection_string = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SERVER};DATABASE={DATABASE};Trusted_Connection=yes;'
db = pyodbc.connect(connection_string)
cursor = db.cursor()

# the base URL to query the books API
API_URL = 'https://openlibrary.org/api/books'

# a list of ISBNs to be added to the catalog
isbn_list = [
    #'978-0201853926',
    #'0201558025',
    #'978-1-93435-645-6',
    #'9780199218462',
    #'978-4915512377',
    #'1593278551',
    #'0811862151',
    #'1844834115',
    #'9780545790352',
    '978-0201633610',
]

# cycle through the list and query the API
for isbn in isbn_list:
    # grab a fresh cursor
    cursor = db.cursor()

    # set up proper search parameters for the API
    isbn_payload = 'ISBN:%s' % isbn
    print(isbn_payload)

    # set all parameters needed to make the request
    params = {
        'bibkeys':isbn_payload,
        'format':'json',
        'jscmd':'data'
    }
    print(f'{API_URL}?bibkeys={params['bibkeys']}&format={params['format']}&jscmd={params['jscmd']}')
    
    # send the request
    r = requests.get(API_URL, params)

    # retreive the book from the JSON data
    book = r.json()[isbn_payload]

    # obtain all information we require to put into our database
    title = book['title'][:100]
    subtitle = book['subtitle'][:100] if 'subtitle' in book else ''
    no_pages = book['number_of_pages'] if 'number_of_pages' in book else 0
    authors = [auth['name'] for auth in book['authors']]
    themes = [subj['name'] for subj in book['subjects']] if 'subjects' in book else []

    # display progress
    print(f' - Title: {title}')
    if subtitle != '': print(f' - Sub Title: {subtitle}')

    # clean up ISBN from input (numbers only to fit the VARCHAR(13) limit)
    isbn_clean = re.sub('[^0-9]', '', isbn)

    # insert data into database

    # Insert book (if it does exist then pass)
    try:
        insert_book_stmt = 'INSERT INTO books(isbn, title, subtitle, no_pages) VALUES(?, ?, ?, ?);'
        cursor.execute(insert_book_stmt, (isbn_clean, title, subtitle, no_pages))
    except pyodbc.IntegrityError as e:
        pass

    # insert authors
    # check if author exists in DB otherwise create and retreive the ID
    insert_author_stmt = 'INSERT INTO authors(author_name) VALUES(?)'
    insert_authors_books_stmt = 'INSERT INTO authors_books VALUES (?, ?)'
    for author in authors:
        print(f' - Author: {author}')
        select_author_id_stmt = 'SELECT id FROM authors WHERE author_name = ?;'
        cursor.execute(select_author_id_stmt, (author))
        result = cursor.fetchone()

        if result is None:
            cursor.execute(insert_author_stmt, (author))
            # Get the ID of the last inserted row using SCOPE_IDENTITY()
            cursor.execute('SELECT SCOPE_IDENTITY();')
            author_id = cursor.fetchone()[0]
        else:
            author_id = result[0]

        # insert author_id into authors_books
        try:
            cursor.execute(insert_authors_books_stmt, (isbn_clean, author_id))
        except pyodbc.IntegrityError as e:
            pass

    # insert themes
    # check if theme exists in DB otherwise create and retreive the ID
    insert_theme_stmt = 'INSERT INTO themes(theme_name) VALUES(?)'
    insert_books_themes_stmt = 'INSERT INTO books_themes VALUES (?, ?)'
    for theme in themes:
        print(f' - Theme: {theme}')
        select_theme_id_stmt = 'SELECT id FROM themes WHERE theme_name = ?;'
        cursor.execute(select_theme_id_stmt, (theme))
        result = cursor.fetchone()

        if result is None:
            cursor.execute(insert_theme_stmt, theme)
            # Get the ID of the last inserted row using SCOPE_IDENTITY()
            cursor.execute('SELECT SCOPE_IDENTITY();')
            theme_id = cursor.fetchone()[0]
        else:
            theme_id = result[0]

        # insert record into books_themes
        try:
            cursor.execute(insert_books_themes_stmt, (isbn_clean, theme_id))
        except pyodbc.IntegrityError as e:
            pass
    
    db.commit()
    cursor.close()

ISBN:978-0201633610
https://openlibrary.org/api/books?bibkeys=ISBN:978-0201633610&format=json&jscmd=data


NameError: name 'ConnectTimeout' is not defined

---
**BONUS**: Using the `book-catalog` Python notebook, add in your own ISBN to be inserted into the database. 

You can also confirm the data is stored by using a SQL SELECT statement. For example:

In [154]:
db = pyodbc.connect(connection_string)
cursor = db.cursor()

# Use the book_catalog database
cursor.execute('USE book_catalog')

# Write and execute the SQL query to select all rows from the 'authors' table
select_query = 'SELECT * FROM authors;'
cursor.execute(select_query)

# Fetch all rows from the executed query
rows = cursor.fetchall()

# Iterate over the rows and print them
for row in rows:
    print(row)

# Close the cursor
cursor.close()

(1, 'George Orwell')
